# Finetuning Granite Speech (LoRA)


In [ ]:
!pip install -U "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec peft bitsandbytes


## Setup

In [1]:
import torch
from pathlib import Path
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"

MANIFEST_DIR = Path("asr_finetune")
OUTPUT_DIR = MANIFEST_DIR / "checkpoint"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("cuda")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

Loading weights:   0%|          | 0/954 [00:00<?, ?it/s]

Model loaded on cuda:0 with dtype torch.bfloat16


## Load the Training and Dev Manifests

In [2]:
import json

MANIFEST_TRAIN = MANIFEST_DIR / "manifest_train.jsonl"
MANIFEST_DEV = MANIFEST_DIR / "manifest_dev.jsonl"

def read_manifest(path):
    with open(path, encoding="UTF-8") as rf:
        return [json.loads(line) for line in rf]

train_data = read_manifest(MANIFEST_TRAIN)[:2000]
dev_data = read_manifest(MANIFEST_DEV)[:100]

print(len(train_data), "train samples,", len(dev_data), "dev samples")
print(train_data[0])

2000 train samples, 100 dev samples
{'audio': '/training-1/mls/mls_english_opus/train/audio/3488/5225/3488_5225_001583.opus', 'text': "it wasn't so terrible dark with the snow on the ground and i was so close to him i could have put an apple in his pocket joe explained who was the man asked moxlow i thought he looked like john north said montgomery", 'prompt': 'Vocabulary: joe, moxlow, john north, montgomery.'}


## Prompt Construction

In [3]:
import torchaudio

def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform

def build_chat_prompt(user_prompt):
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{user_prompt}"},
    ]
    return processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

DEFAULT_PROMPT = "can you transcribe the speech into a written format?"

def build_user_prompt(sample):
    prompt = sample.get("prompt")
    if prompt:
        return f"{DEFAULT_PROMPT} Keywords: {prompt}"
    return DEFAULT_PROMPT

## LoRA Configuration (PEFT)

In [4]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,259,840 || all params: 2,317,401,436 || trainable%: 0.1838


## Collate Function (Audio + Prompt + Target Text)

In [5]:
def collate_fn(batch):
    wavs = [load_audio(sample["audio"]) for sample in batch]
    prompts = [build_chat_prompt(build_user_prompt(sample)) for sample in batch]

    inputs = processor(prompts, wavs, return_tensors="pt", padding=True)

    eos_id = processor.tokenizer.eos_token_id   # the model must learn to stop
    target_ids = []
    for sample in batch:
        t = processor.tokenizer(sample["text"], add_special_tokens=False,
                                return_tensors="pt").input_ids[0]
        target_ids.append(torch.cat([t, torch.tensor([eos_id], dtype=t.dtype)]))
    batch_size, prompt_len = inputs["input_ids"].shape
    max_len = prompt_len + max(len(t) for t in target_ids)
    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0

    input_ids = torch.full((batch_size, max_len), pad_id, dtype=inputs["input_ids"].dtype)
    attention_mask = torch.zeros((batch_size, max_len), dtype=inputs["attention_mask"].dtype)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)

    input_ids[:, :prompt_len] = inputs["input_ids"]
    attention_mask[:, :prompt_len] = inputs["attention_mask"]
    for i, t in enumerate(target_ids):
        input_ids[i, prompt_len:prompt_len + len(t)] = t
        attention_mask[i, prompt_len:prompt_len + len(t)] = 1
        labels[i, prompt_len:prompt_len + len(t)] = t

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        **{k: v for k, v in inputs.items() if k not in ("input_ids", "attention_mask")},
    }

batch = collate_fn(train_data[:2])
mask = batch["labels"] != -100
print(processor.tokenizer.decode(batch["labels"][0][mask[0]]))

it wasn't so terrible dark with the snow on the ground and i was so close to him i could have put an apple in his pocket joe explained who was the man asked moxlow i thought he looked like john north said montgomery<|end_of_text|>


## Training Configuration

In [6]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR.as_posix(),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=2,
    eval_strategy="epoch",
    logging_steps=10,
    logging_first_step=True,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to=[],
    remove_unused_columns=False,
    disable_tqdm=False,
)


## Evaluation (Dev WER)

In [7]:
import jiwer
from tqdm import tqdm

def compute_dev_wer():
    model.eval()
    wers = []
    for sample in tqdm(dev_data):
        wav = load_audio(sample["audio"])
        text_prompt = build_chat_prompt(build_user_prompt(sample))
        inputs = processor(text_prompt, wav, device=model.device, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1,
                                 use_cache=True)
        hyp = processor.tokenizer.decode(
            out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
        wers.append(jiwer.wer(sample["text"], hyp))
    model.train()
    return sum(wers) / len(wers)



In [ ]:
print("Dev WER before training:", compute_dev_wer())

## Train

In [9]:
from transformers import Trainer, TrainerCallback


class ConsoleLogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"step {state.global_step}: {logs}")


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=dev_data,
    data_collator=collate_fn,
    callbacks=[ConsoleLogCallback()],
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,1.895799,0.622576
2,1.262327,0.491278


step 1: {'loss': 18.471729278564453, 'grad_norm': 53.26190948486328, 'learning_rate': 1e-05, 'epoch': 0.002}
step 10: {'loss': 16.413869222005207, 'grad_norm': 66.69324493408203, 'learning_rate': 9.91e-06, 'epoch': 0.02}
step 20: {'loss': 7.894224548339844, 'grad_norm': 4.558482646942139, 'learning_rate': 9.810000000000001e-06, 'epoch': 0.04}
step 30: {'loss': 6.71805419921875, 'grad_norm': 17.092683792114258, 'learning_rate': 9.71e-06, 'epoch': 0.06}
step 40: {'loss': 5.371460342407227, 'grad_norm': 4.505098342895508, 'learning_rate': 9.610000000000001e-06, 'epoch': 0.08}
step 50: {'loss': 10.944291687011718, 'grad_norm': 21.852201461791992, 'learning_rate': 9.51e-06, 'epoch': 0.1}
step 60: {'loss': 9.755220794677735, 'grad_norm': 219.03355407714844, 'learning_rate': 9.41e-06, 'epoch': 0.12}
step 70: {'loss': 8.19740982055664, 'grad_norm': 7.7549591064453125, 'learning_rate': 9.31e-06, 'epoch': 0.14}
step 80: {'loss': 6.904836273193359, 'grad_norm': 15.255721092224121, 'learning_rate'

TrainOutput(global_step=1000, training_loss=2.9533160161972045, metrics={'train_runtime': 1414.6179, 'train_samples_per_second': 2.828, 'train_steps_per_second': 0.707, 'total_flos': 1.0818534692634768e+16, 'train_loss': 2.9533160161972045, 'epoch': 2.0})

In [10]:
print("Dev WER AFTER training:", compute_dev_wer())

100%|██████████| 100/100 [04:04<00:00,  2.45s/it]

Dev WER AFTER training: 0.032374808563856515


## Write Out the Checkpoint (LoRA Merged into the Base Model)

In [14]:
merged_model = model.merge_and_unload()   # merge LoRA weights into the base model
merged_model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Checkpoint written to {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint written to asr_finetune/checkpoint
